In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from mne.decoding import CSP
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
import sys
import os
import mlflow
import mlflow.sklearn
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join('..')))

from src.train_BCICIV import train_BCICIV
from src.load_data_BCICIV import load_all_subjects
from src.preprocess import laplacian_filter


mne.set_log_level('WARNING')

## LOAD TRAIN DATA FOR BCICIV DATASET

In [2]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [3]:
split_index = 1006  

X_train = data['X'][:split_index]
y_train = data['y'][:split_index]
subject_ids_train = data['subject_ids'][:split_index]

X_test = data['X'][split_index:]
y_test = data['y'][split_index:]
subject_ids_test = data['subject_ids'][split_index:]

In [4]:
channels_to_use = [3, 7] # C3 and C4

data_2ch = data.copy()
X_2ch_train = data_2ch['X'][:split_index, channels_to_use, :]
X_2ch_test = data_2ch['X'][split_index:, channels_to_use, :]

In [5]:
print("Data shapes:")
print("Original data:", X_train.shape)
print("2-channel data train:", X_2ch_train.shape)
print("2-channel data test:", X_2ch_test.shape)

Data shapes:
Original data: (1006, 11, 1000)
2-channel data train: (1006, 2, 1000)
2-channel data test: (288, 2, 1000)


# INITIAL MODELS NO PREPROCESSING (11 channels)

## CSP + LDA MODEL

In [6]:
pipe1 = Pipeline([
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [7]:
loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(X_train, y_train, subject_ids_train, pipe1, loso, param_grid1)
print(f"Best score: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 8.2e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Estimated rank (data): 11
    data: 

## CSP + SVM MODEL

In [8]:
pipe2 = Pipeline([
    ('csp', CSP(log=True, norm_trace=False)),
    ('clf', SVC(kernel='rbf'))
])

param_grid2 = {
    'csp__n_components': [2, 4, 6],
    'csp__reg': [None, 'ledoit_wolf', 'oas'],
    'clf__C': [0.1, 1, 10],
    'clf__gamma': ['scale', 'auto']
}

In [9]:
loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(X_train, y_train, subject_ids_train, pipe2, loso, param_grid2)
print(f"Best score: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 8.2e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Using tolerance 7.9e-05 (2.2e-16 eps * 11 dim * 3.2e+10  max singular value)
    Using tolerance 8e-05 (2.2e-16 eps * 11 dim * 3.3e+10  max singular value)
    Estimated rank (data): 11
    data: 

# INITIAL MODELS NO PREPROCESSING (2 channels)

## CSP + LDA MODEL

In [15]:
loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(X_2ch_train, y_train, subject_ids_train, pipe1, loso, param_grid1)
print(f"Best score: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.3e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.4e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
    Using tolerance 6.1e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.2e-0

## CSP + SVM MODEL

In [16]:
loso = LeaveOneGroupOut()

model, params, best_score = train_BCICIV(X_2ch_train, y_train, subject_ids_train, pipe2, loso, param_grid2)
print(f"Best score: {best_score}")

Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
Computing rank from data with rank=None
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
    Using tolerance 6.4e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Using tolerance 6.1e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
Reducing data rank from 2 -> 2
Estimating class=0 covariance using EMPIRICAL
Done.
    Using tolerance 6.2e-06 (2.2e-16 eps * 2 dim * 1.4e+10  max singular va

# EEGNet